In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
#from tpch import load_lineitem, load_orders, load_supplier, load_nation, q21
import pandas as pd
DATA_ROOT = "/home/dias-benchmarks/tpch/data"
STORAGE_OPTS = {}



In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)


In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###


lineitem_filtered = lineitem.loc[
    :, ["L_ORDERKEY", "L_SUPPKEY", "L_RECEIPTDATE", "L_COMMITDATE"]
]

# Keep all rows that have another row in linetiem with the same orderkey and different suppkey
lineitem_orderkeys = (
    lineitem_filtered.loc[:, ["L_ORDERKEY", "L_SUPPKEY"]]
    .groupby("L_ORDERKEY", as_index=False, sort=False)["L_SUPPKEY"]
    .nunique()
)
lineitem_orderkeys.columns = ["L_ORDERKEY", "nunique_col"]
lineitem_orderkeys = lineitem_orderkeys[lineitem_orderkeys["nunique_col"] > 1]
lineitem_orderkeys = lineitem_orderkeys.loc[:, ["L_ORDERKEY"]]

# Keep all rows that have l_receiptdate > l_commitdate
lineitem_filtered = lineitem_filtered[
    lineitem_filtered["L_RECEIPTDATE"] > lineitem_filtered["L_COMMITDATE"]
]
lineitem_filtered = lineitem_filtered.loc[:, ["L_ORDERKEY", "L_SUPPKEY"]]

# Merge Filter + Exists
lineitem_filtered = lineitem_filtered.merge(
    lineitem_orderkeys, on="L_ORDERKEY", how="inner"
)

# Not Exists: Check the exists condition isn't still satisfied on the output.
lineitem_orderkeys = lineitem_filtered.groupby(
    "L_ORDERKEY", as_index=False, sort=False
)["L_SUPPKEY"].nunique()
lineitem_orderkeys.columns = ["L_ORDERKEY", "nunique_col"]
lineitem_orderkeys = lineitem_orderkeys[lineitem_orderkeys["nunique_col"] == 1]
lineitem_orderkeys = lineitem_orderkeys.loc[:, ["L_ORDERKEY"]]

# Merge Filter + Not Exists
lineitem_filtered = lineitem_filtered.merge(
    lineitem_orderkeys, on="L_ORDERKEY", how="inner"
)

orders_filtered = orders.loc[:, ["O_ORDERSTATUS", "O_ORDERKEY"]]
orders_filtered = orders_filtered[orders_filtered["O_ORDERSTATUS"] == "F"]
orders_filtered = orders_filtered.loc[:, ["O_ORDERKEY"]]
total = lineitem_filtered.merge(
    orders_filtered, left_on="L_ORDERKEY", right_on="O_ORDERKEY", how="inner"
)
total = total.loc[:, ["L_SUPPKEY"]]

supplier_filtered = supplier.loc[:, ["S_SUPPKEY", "S_NATIONKEY", "S_NAME"]]
total = total.merge(
    supplier_filtered, left_on="L_SUPPKEY", right_on="S_SUPPKEY", how="inner"
)
total = total.loc[:, ["S_NATIONKEY", "S_NAME"]]
nation_filtered = nation.loc[:, ["N_NAME", "N_NATIONKEY"]]
nation_filtered = nation_filtered[nation_filtered["N_NAME"] == "SAUDI ARABIA"]
total = total.merge(
    nation_filtered, left_on="S_NATIONKEY", right_on="N_NATIONKEY", how="inner"
)
total = total.loc[:, ["S_NAME"]]
total = total.groupby("S_NAME", as_index=False, sort=False).size()
total.columns = ["S_NAME", "NUMWAIT"]
total = total.sort_values(by=["NUMWAIT", "S_NAME"], ascending=[False, True])


In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

total.info()